# 04 · Flow Optional —— 双向拉链：x 和 z 精确互逆

**家族位置**：07 生成式模型第 4 站（可选拓展）。AE 压缩有损、VAE 近似后验、GAN 无似然、Diffusion 链式去噪——Flow 走第四条路：**可逆变换**，x→z 编码、z→x 解码、log-det 精确算概率。

**学习目标**：理解 RealNVP 仿射耦合层（前向/逆向公式、log-det）；NLL=先验+体积变化；往返一致性验证；与 VAE 的 ELBO 对比。

## 1. 原理：拉链能正反拉

### 通俗理解

**一句话**：AE 是单向复印（压了就回不去），VAE 是近似（后验猜的），GAN 是黑箱（概率都算不出），Flow 是**双向拉链**——x 拉成 z，z 拉回 x，分毫不差，还能精确说出每张图在模型眼里有多大概率。

### 结构账

```
耦合层： x=(x1,x2) → s,t=NN(x1) → y2=x2·exp(s)+t → y=(x1,y2)；交换 halves 再来
逆向： x2=(y2-t)·exp(-s)，s/t 同一组参数反着用
log-det = Σs（三角雅可比，对角线就是 exp(s)）
NLL = -[log N(z;0,I) + Σlog-det]，784 维展平 + 去量化噪声
```

去量化：MNIST 离散 0-255 灰度，加 U(0,1)/256 抖动再 logit 风格缩放到 [-1,1]，连续流才能建模。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import load_mnist_local
from common.models import GlowLite
from common.utils import set_seed,setup_chinese_font,count_params,show_mnist
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
Xtr,ytr,Xva,yva,Xte,yte=load_mnist_local(4000,800,seed=0)
print(f'train {tuple(Xtr.shape)} | dequant+logit 预处理见 cell2')

def preprocess(x):
    x01=(x*0.3081+0.1307).clamp(0,1)
    x01=(x01*255.0+torch.rand_like(x01))/256.0
    return (x01-0.5)/0.5  # [-1,1]

def deprocess(x):
    return (x*0.5+0.5).clamp(0,1)
loader=DataLoader(TensorDataset(preprocess(Xtr),ytr),batch_size=128,shuffle=True)
vloader=DataLoader(TensorDataset(preprocess(Xva),yva),batch_size=512)
fig,ax=plt.subplots(1,2,figsize=(8,3.0))
ax[0].text(0.5,0.6,'x=(x1,x2)\n→ s,t=NN(x1)\n→ y2=x2·exp(s)+t',ha='center',fontsize=11); ax[0].axis('off'); ax[0].set_title('前向：拉上拉链')
ax[1].text(0.5,0.6,'y=(y1,y2)\n→ x2=(y2-t)·exp(-s)\n→ 回到 x',ha='center',fontsize=11); ax[1].axis('off'); ax[1].set_title('逆向：拉开拉链')
plt.suptitle('RealNVP 仿射耦合：同一组 s/t，正反各用一次'); plt.tight_layout()
plt.savefig(FIGS/'fig0_coupling.png',dpi=150,bbox_inches='tight'); plt.show()

## 2. 训练：最大化精确似然

In [ ]:
flow=GlowLite(dim=784,n_couplings=6,hidden=256)
print(f'GlowLite params={count_params(flow)} | 6 耦合层×(256×256 MLP)')
opt=torch.optim.Adam(flow.parameters(),lr=1e-3)
hist=[]
for ep in range(1,9):
    flow.train(); tot=0
    for xb,yb in loader:
        loss=flow.nll(xb.flatten(1)); opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(flow.parameters(),1.0); opt.step(); tot+=loss.item()*len(xb)
    hist.append(tot/len(loader.dataset))
    with torch.no_grad():
        flow.eval(); va=sum(flow.nll(xb.flatten(1)).item()*len(xb) for xb,yb in vloader)/len(vloader.dataset)
    print(f'epoch {ep:02d} | train NLL {hist[-1]:.1f} | val NLL {va:.1f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2)); ax.plot(hist,marker='o',color='#4C72B0')
ax.set_xlabel('epoch'); ax.set_ylabel('NLL (nats/图)'); ax.set_title('Flow 训练：精确似然单调优化（VAE 只能优化下界）')
plt.tight_layout(); plt.savefig(FIGS/'fig1_flow_loss.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 往返一致性：x→z→x̂ 分毫不差

In [ ]:
with torch.no_grad():
    flow.eval(); xb=preprocess(Xte[:8])
    z,_=flow(xb.flatten(1)); xhat,_=flow(z,reverse=True)
    err=(xhat-xb.flatten(1)).abs().max().item()
    print(f'roundtrip max|Δ|={err:.2e}（float32 可逆极限，非学习结果）')
fig,ax=plt.subplots(2,8,figsize=(10,2.8))
for j in range(8):
    ax[0,j].imshow(deprocess(xb[j]).numpy()[0],cmap='gray'); ax[0,j].axis('off')
    ax[1,j].imshow(deprocess(xhat[j].view(1,28,28)).numpy()[0],cmap='gray'); ax[1,j].axis('off')
ax[0,0].set_ylabel('原图'); ax[1,0].set_ylabel('往返'); plt.suptitle(f'x→z→x̂ 逐像素一致（max Δ={err:.1e}）')
plt.tight_layout(); plt.savefig(FIGS/'fig2_roundtrip.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. 采样与似然直方图：z~N(0,I) 拉回 x

In [ ]:
with torch.no_grad():
    flow.eval(); samp=flow.sample(16,temperature=0.7)
    nll_tr=torch.stack([flow.nll(preprocess(Xtr[i:i+256]).flatten(1)).detach() for i in range(0,2000,256)]).mean().item()
    nll_te=torch.stack([flow.nll(preprocess(Xte[i:i+256]).flatten(1)).detach() for i in range(0,2000,256)]).mean().item()
    print(f'train NLL={nll_tr:.1f} test NLL={nll_te:.1f}（nats/图；越小越好，精确值非下界）')
fig,ax=plt.subplots(2,8,figsize=(10,3.0))
for a,x in zip(ax.flat,samp): a.imshow(deprocess(x).numpy()[0],cmap='gray'); a.axis('off')
plt.suptitle('Flow 采样（T=0.7）：可辨但偏糊——展平流丢了空间局部性'); plt.tight_layout()
plt.savefig(FIGS/'fig3_samples.png',dpi=150,bbox_inches='tight'); plt.show()
with torch.no_grad():
    nlls=torch.cat([flow.nll(preprocess(Xte[i:i+512]).flatten(1)).detach().unsqueeze(0) for i in range(0,2000,512)]).flatten()
fig,ax=plt.subplots(figsize=(6,3.2)); ax.hist(nlls.numpy(),bins=30,color='#4C72B0',alpha=0.8)
ax.axvline(nll_te,color='red',ls='--',label=f'mean {nll_te:.0f}'); ax.legend()
ax.set_xlabel('NLL (nats/图)'); ax.set_title('测试集似然直方图：每张图都有精确分数（GAN 给不出）')
plt.tight_layout(); plt.savefig(FIGS/'fig4_nll_hist.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. 总结

Flow 闭环：耦合层可逆 → log-det 精确 → NLL 直接优化 → 往返一致 + 采样 + 似然直方图。与 VAE 的 ELBO（下界）对比：Flow 算的是真值，代价是可逆约束限制表达 + 展平丢空间。07 家族至此 4/4 全满：AE 复现 → GAN 对抗 → Diffusion 去噪 → Flow 可逆。